# LangChain Tool Calling with OpenAI
Ví dụ định nghĩa, bind và gọi tools bằng LangChain.

In [ ]:
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import (
    ArxivLoader,
    WebBaseLoader,
    WikipediaLoader,
)
import arxiv
import requests
import time
import wikipedia
import xml.etree.ElementTree as ET
from urllib.parse import quote
from langchain_core.documents import Document

load_dotenv()

In [ ]:
# wikipedia 1.4.0 builds an HTTP API URL; Wikimedia requires HTTPS.


# arxiv 1.x also defaults to HTTP, which is blocked by many environments.

## Define tools

In [ ]:
@tool
def reverse_string(input_string: str) -> str:

In [ ]:
def format_documents(documents, max_chars: int = 4000) -> str:
    """Convert loaded documents into compact text for the model."""
    if not documents:
        return "Không tìm thấy tài liệu."
    sections = []
    for index, document in enumerate(documents, start=1):
        title = document.metadata.get("title", f"Tài liệu {index}")
        source = document.metadata.get("source", document.metadata.get("entry_id", "Không rõ nguồn"))
        sections.append(f"Tiêu đề: {title}\nNguồn: {source}\nNội dung:\n{document.page_content[:max_chars]}")
    return "\n\n---\n\n".join(sections)

def load_with_retry(loader_factory, attempts: int = 3, delay: float = 1.0):
    """Retry transient loader/network failures, then raise the last error."""
    last_error = None
    for attempt in range(attempts):
        try:
            documents = loader_factory().load()
            if documents:
                return documents
        except Exception as exc:
            last_error = exc
        if attempt < attempts - 1:
            time.sleep(delay * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

@tool
def search_wikipedia(query: str, top_k: int = 2) -> str:
    """Tìm thông tin bách khoa trên Wikipedia."""
    try:
        # Call WikipediaLoader

    except Exception as loader_exc:
        # Fallback for wikipedia 1.4.0/API responses that are not valid JSON.
        try:
            # Avoid another API request (and possible 429): load the likely article URL directly.
            article_url = f"https://vi.wikipedia.org/wiki/{quote(query.strip().replace(' ', '_'))}"
            return format_documents(WebBaseLoader(web_paths=(article_url,), requests_kwargs={"timeout": 20}).load())
        except Exception as fallback_exc:
            return (f"Wikipedia hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def search_arxiv(query: str, top_k: int = 2) -> str:
    """Tìm các bài nghiên cứu khoa học trên arXiv."""
    try:
        # Call ArxivLoader

    except Exception as loader_exc:
        # Fallback when arxiv 1.x/feedparser cannot expose the HTTP status.
        try:
            response = requests.get(
                "https://export.arxiv.org/api/query",
                params={"search_query": query, "start": 0, "max_results": top_k},
                headers={"User-Agent": "LangChainToolCallingNotebook/1.0 (educational use)"},
                timeout=30,
            )
            response.raise_for_status()
            root = ET.fromstring(response.content)
            ns = {"atom": "http://www.w3.org/2005/Atom"}
            documents = []
            for entry in root.findall("atom:entry", ns):
                title = " ".join((entry.findtext("atom:title", default="Untitled", namespaces=ns)).split())
                summary = " ".join((entry.findtext("atom:summary", default="", namespaces=ns)).split())
                source = entry.findtext("atom:id", default="", namespaces=ns)
                documents.append(Document(page_content=summary, metadata={"title": title, "source": source}))
            return format_documents(documents)
        except Exception as fallback_exc:
            return (f"arXiv hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def load_web_page(url: str) -> str:
    """Tải và đọc nội dung từ một URL cụ thể."""
    try:
        # Call WebBaseLoader

    except Exception as exc:
        return f"Không tải được trang web: {type(exc).__name__}: {exc}"

# print(search_wikipedia.invoke({"query": "Trí tuệ nhân tạo", "top_k": 1})[:500])

In [ ]:
# Multiply tool

## OpenAI model

## Binding and executing tools

## Additional tests
Các kiểm thử dưới đây xác nhận tool cơ bản, loader và OpenAI tool calling.

In [ ]:
# Run every loader and display its full result
loader_cases = {
    "wikipedia": (search_wikipedia, {"query": "Trí tuệ nhân tạo", "top_k": 1}),
    "arxiv": (search_arxiv, {"query": "cat:cs.AI AND ti:agent", "top_k": 1}),
    "web": (load_web_page, {"url": "https://example.com"}),
}
loader_results = {}
for name, (loader_tool, arguments) in loader_cases.items():
    output = loader_tool.invoke(arguments)
    loader_results[name] = output
    print(f"\n{'=' * 20} {name.upper()} {'=' * 20}")
    print(output)

loader_results

In [ ]:
# Automatic routing test with explicit source requests
routing_cases = [
    ("Chỉ dùng Wikipedia để tìm thông tin về Hà Nội.", "search_wikipedia"),
    ("Chỉ dùng arXiv để tìm nghiên cứu về large language model agents.", "search_arxiv"),
    ("Hãy đọc URL https://example.com bằng công cụ tải trang web.", "load_web_page"),
]
for prompt, expected_name in routing_cases:
    routed = model_with_tools.invoke(prompt)
    assert routed.tool_calls, f"No tool call for: {prompt}"
    assert routed.tool_calls[0]["name"] == expected_name, routed.tool_calls
    print(f"PASS: automatic routing -> {expected_name}")